# E2 — Python → Gemini → MySQL
Execute as células na ordem. Use somente os casos sintéticos fornecidos. Este notebook não estima risco clínico. A execução real ainda precisa ser validada nos dois ambientes.

**LOCAL:** Colab conectado ao Jupyter do seu computador, MySQL local.
**NUVEM:** Colab hospedado pelo Google, MySQL externo com TLS e acesso autorizado. Não use localhost para tentar alcançar o computador de casa.

In [ ]:
MODO = "LOCAL"  # LOCAL ou NUVEM
from pathlib import Path
import os, sys, subprocess, json
if MODO == "LOCAL":
    PROJETO = Path(r"C:\Users\ralbertijuni\Desktop\tia_api\entrega_e2")
elif MODO == "NUVEM":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJETO = Path("/content/drive/MyDrive/TIA/entrega_e2")
else:
    raise ValueError("MODO deve ser LOCAL ou NUVEM")
assert (PROJETO / "src/main.py").is_file(), "Confira o caminho da pasta do projeto."
os.chdir(PROJETO)
print("Pasta do projeto encontrada. Modo:", MODO)

Pasta do projeto encontrada. Modo: LOCAL


## Dependências e validação local
A instalação abaixo ocorre no ambiente conectado ao Colab. Não reutiliza o venv antigo da aplicação.

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=True)
subprocess.run([sys.executable, "-B", "-m", "unittest", "discover", "-s", "tests", "-v"], check=True)
subprocess.run([sys.executable, "-B", "-m", "src.main", "--validar-dados"], check=True)

CompletedProcess(args=['C:\\Users\\ralbertijuni\\Desktop\\tia_api\\entrega_e2\\.venv-colab\\Scripts\\python.exe', '-B', '-m', 'src.main', '--validar-dados'], returncode=0)

## Credenciais
LOCAL: preencha o `.env` da pasta entrega_e2 no computador. Não cole valores nesta célula.
NUVEM: no painel Secrets (ícone de chave), cadastre os nomes abaixo e autorize o notebook. O certificado CA do provedor é público e deve ficar no caminho configurado; nenhuma chave privada deve ser enviada.

In [ ]:
if MODO == "NUVEM":
    from google.colab import userdata
    for nome in ("GEMINI_API_KEY", "GEMINI_MODEL", "MYSQL_HOST", "MYSQL_PORT",
                 "MYSQL_USER", "MYSQL_PASSWORD", "MYSQL_DATABASE"):
        try:
            os.environ[nome] = userdata.get(nome)
        except Exception:
            raise RuntimeError("Configure e autorize o Secret " + nome) from None
    host = os.environ["MYSQL_HOST"].strip().lower()
    if host in ("localhost", "127.0.0.1", "::1", ""):
        raise ValueError("No modo NUVEM, use o endereço real do MySQL remoto.")
    os.environ["MYSQL_SSL_CA"] = str(PROJETO / "certificados/ca.pem")
    os.environ["MYSQL_REQUIRE_TLS"] = "1"
    assert Path(os.environ["MYSQL_SSL_CA"]).is_file(), "Adicione o certificado CA fornecido pelo provedor."
else:
    # Limpa opções de nuvem caso esta sessão tenha sido usada em outro modo.
    os.environ.pop("MYSQL_REQUIRE_TLS", None)
    os.environ.pop("MYSQL_SSL_CA", None)
print("Configuração selecionada; nenhum segredo exibido.")

Configuração selecionada; nenhum segredo exibido.


## Conferir acesso ao MySQL
O banco deve existir e o usuário precisa de SELECT, INSERT e CREATE para a baseline. Esta célula não chama o Gemini.

In [ ]:
from src.config import carregar
from src.database import Database
config = carregar()
db = None
try:
    db = Database(config)
    db.inicializar()
    with db.conn.cursor() as cur:
        cur.execute("SELECT DATABASE(), VERSION()")
        print("Banco e versão:", cur.fetchone())
        cur.execute("SHOW SESSION STATUS LIKE 'Ssl_cipher'")
        tls = cur.fetchone()
        if MODO == "NUVEM" and (not tls or not tls[1]):
            raise RuntimeError("Conexão sem TLS: não prossiga.")
        print("Conexão MySQL confirmada.")
except Exception as exc:
    raise RuntimeError("Falha MySQL: " + type(exc).__name__ + ". Confira rede, usuário e certificado.") from None
finally:
    if db is not None:
        db.close()

Banco e versão: ('tia_e2', '26.7.0')
Conexão MySQL confirmada.


In [ ]:
from dotenv import load_dotenv
from src.config import ROOT
import os

load_dotenv(ROOT / ".env", override=True)

print("Modelo carregado:", os.getenv("GEMINI_MODEL"))

Modelo carregado: gemini-3.1-flash-lite


## Executar pipeline real
Faz três chamadas ao Gemini, consumindo cota, e grava os resultados. Execute conscientemente: cada execução cria novos UUIDs. Falhas não são substituídas por resultados fictícios.

In [ ]:
import subprocess, sys

antes = set((PROJETO / "docs/evidencias").glob("*.json"))

resultado = subprocess.run(
    [sys.executable, "-B", "-m", "src.main"],
    cwd=PROJETO,                 # garante que "-m src.main" encontra o pacote src/
    capture_output=True,
    text=True,
)

print("--- STDOUT ---")
print(resultado.stdout)
print("--- STDERR ---")
print(resultado.stderr)

novos = set((PROJETO / "docs/evidencias").glob("*.json")) - antes

if resultado.returncode != 0:
    raise RuntimeError("Pipeline falhou. Consulte o STDERR acima; não declarar E2 concluída.")
assert len(novos) == 1, "Não foi possível identificar o relatório desta execução."
relatorio_path = novos.pop()
relatorio = json.loads(relatorio_path.read_text(encoding="utf-8"))
assert relatorio["sucesso"] and len(relatorio["casos"]) == 3
print(json.dumps(relatorio, indent=2, ensure_ascii=False))

--- STDOUT ---
Relatório: C:\Users\ralbertijuni\Desktop\tia_api\entrega_e2\docs\evidencias\716833c5-e504-4e4e-af0b-b6eb14c2a480.json. Sucesso integral: True

--- STDERR ---

{
  "inicio_utc": "2026-10-07T20:53:36.334361+00:00",
  "casos": [
    {
      "caso_id": "SINT_001",
      "execucao_id": "2c84e7fc-6579-4b9d-b351-b6ffaa4a7cf9",
      "status": "PERSISTIDO_E_RELIDO",
      "modelo": "gemini-3.1-flash-lite",
      "resposta_sha256": "172fc924320ed90ec90e350ff922e4d4c1bbf35d23d29d742b10d358d2d51b0d"
    },
    {
      "caso_id": "SINT_002",
      "execucao_id": "92b9cdab-e52d-4ce0-817d-bddf91f3a785",
      "status": "PERSISTIDO_E_RELIDO",
      "modelo": "gemini-3.1-flash-lite",
      "resposta_sha256": "d18dda9539f5c97bd222f0e4bec76d21a96f3c06ae2632e7fb42ab41540ceaf7"
    },
    {
      "caso_id": "SINT_003",
      "execucao_id": "901e8a9d-87a6-452c-af3e-b6d0ddde9e26",
      "status": "PERSISTIDO_E_RELIDO",
      "modelo": "gemini-3.1-flash-lite",
      "resposta_sha256": "92cdb51

In [50]:
antes = set((PROJETO / "docs/evidencias").glob("*.json"))
resultado = subprocess.run([sys.executable, "-B", "-m", "src.main"])
novos = set((PROJETO / "docs/evidencias").glob("*.json")) - antes
if resultado.returncode != 0:
    raise RuntimeError("Pipeline falhou. Consulte o relatório; não declarar E2 concluída.")
assert len(novos) == 1, "Não foi possível identificar o relatório desta execução."
relatorio_path = novos.pop()
relatorio = json.loads(relatorio_path.read_text(encoding="utf-8"))
assert relatorio["sucesso"] and len(relatorio["casos"]) == 3
print(json.dumps(relatorio, indent=2, ensure_ascii=False))

{
  "inicio_utc": "2026-10-07T20:54:08.047127+00:00",
  "casos": [
    {
      "caso_id": "SINT_001",
      "execucao_id": "53d7c8a6-025c-4cbe-b857-dd2628e56485",
      "status": "PERSISTIDO_E_RELIDO",
      "modelo": "gemini-3.1-flash-lite",
      "resposta_sha256": "172fc924320ed90ec90e350ff922e4d4c1bbf35d23d29d742b10d358d2d51b0d"
    },
    {
      "caso_id": "SINT_002",
      "execucao_id": "3d29f78c-0219-40e1-bcb1-2296694ccad5",
      "status": "PERSISTIDO_E_RELIDO",
      "modelo": "gemini-3.1-flash-lite",
      "resposta_sha256": "d18dda9539f5c97bd222f0e4bec76d21a96f3c06ae2632e7fb42ab41540ceaf7"
    },
    {
      "caso_id": "SINT_003",
      "execucao_id": "4b8d7d21-0c26-4e30-8b53-98eb0f86728b",
      "status": "PERSISTIDO_E_RELIDO",
      "modelo": "gemini-3.1-flash-lite",
      "resposta_sha256": "92cdb517ff49867a7c308c068a0eb75905c36622edbcadc579d9b1589f932a73"
    }
  ],
  "sucesso": true
}


## Mostrar exatamente os registros desta execução
A apresentação verifica a persistência dos mesmos UUIDs do relatório.

In [ ]:
db = Database(carregar())
try:
    with db.conn.cursor(dictionary=True) as cur:
        for caso in relatorio["casos"]:
            cur.execute("SELECT execucao_id, caso_id, modelo, resposta_json, resposta_sha256 FROM baseline_execucao WHERE execucao_id=%s",
                        (caso["execucao_id"],))
            row = cur.fetchone()
            assert row and row["resposta_sha256"] == caso["resposta_sha256"]
            print(json.dumps(row, indent=2, ensure_ascii=False))
finally:
    db.close()
print("Três resultados conferidos no banco.")

{
  "execucao_id": "53d7c8a6-025c-4cbe-b857-dd2628e56485",
  "caso_id": "SINT_001",
  "modelo": "gemini-3.1-flash-lite",
  "resposta_json": "{\"resumo\": \"O conjunto de dados apresenta informações de um paciente sintético, incluindo identificador único, origem da fonte, idade, código de diagnóstico principal segundo a CID-10 (Insuficiência Cardíaca Congestiva) e a contagem de internações hospitalares nos últimos doze meses.\", \"limitacoes\": [\"Ausência de variáveis clínicas cruciais como comorbidades, sinais vitais e resultados de exames laboratoriais.\", \"Falta de dados socioeconômicos e de suporte social, que são determinantes para a readmissão hospitalar.\", \"Inexistência de informações sobre o histórico de medicações e adesão ao tratamento pós-alta.\", \"Dados puramente sintéticos que não refletem a complexidade, a variabilidade ou as relações causais de pacientes reais.\", \"Ausência de dados temporais detalhados, como a duração da internação ou o intervalo exato entre os epi

## Encerrar
Salve o notebook e o relatório no Drive. Antes de compartilhar, revise as saídas. Nunca compartilhe o token do Jupyter, `.env` ou Secrets. O sucesso aqui comprova integração, não acurácia clínica.

Na banca: abrir notebook → NUVEM → conectar → executar em ordem → mostrar registros no MySQL. Ensaie antes na rede do curso; acesso ao banco remoto pode ser bloqueado.